# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset is described by a Croissant schema and provides structured clinical, pathological, and biomarker data for 77 cancer survivors with a second primary colorectal cancer.

### Dataset Source
The dataset's Croissant schema is hosted at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure that mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully!")
print(f"Title: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview

Let's list the main record sets and their fields, referencing each by its `@id` as per Croissant conventions.

In [ ]:
# The 'recordSets' method provides available record set @ids in the dataset
record_sets = list(dataset.record_sets)
print("Record sets (@ids):")
for rs_id in record_sets:
    print(f"- {rs_id}")

# Let's inspect the fields for each record set
for rs_id in record_sets:
    print(f"\nRecord set: {rs_id}")
    # Get the field definitions
    fields = dataset.get_record_set_fields(rs_id)
    for field in fields:
        print(f"  Field: {field.get('@id', field.get('name'))} (name: {field.get('name')})")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame. For this dataset, the main record set will typically contain the clinical observation records. You're encouraged to inspect other record sets as well if available.

_All references use the respective `@id` values printed above._

In [ ]:
# Extract data from each record set as DataFrames

dataframes = {}

for rs_id in record_sets:
    print(f"Loading records for record set '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  - Loaded {len(df)} records; Columns (@id): {list(df.columns)}\n")
# For demonstration, select the first record set as main
main_record_set = record_sets[0] if record_sets else None

if main_record_set:
    print(f"Example columns in '{main_record_set}':\n{dataframes[main_record_set].columns.tolist()}")
    display(dataframes[main_record_set].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)

Let's process and explore the data:
- **Filtering** (e.g. on a numeric field such as age, diagnosis interval, or tumor size)
- **Normalization** (z-scoring a numeric column)
- **Grouping** (e.g. by a biomarker status or anatomical site)

_All field references are by their exact `@id` strings._

In [ ]:
# Choose field @ids (fill in with actual values based on overview output above)

# Replace these example @ids with your dataset's actual field @ids:
numeric_field_id = None
group_field_id = None

# Infer candidates by examining columns from previous step
main_df = dataframes[main_record_set].copy()

# Try to automatically select a first numeric column and a likely group column
for col in main_df.columns:
    # Try to select a numeric column
    if main_df[col].dtype in ['int64', 'float64'] and numeric_field_id is None:
        numeric_field_id = col
    # Look for common grouping fields
    if 'status' in col.lower() or 'type' in col.lower() or 'sex' in col.lower() or 'site' in col.lower():
        group_field_id = col

if numeric_field_id is None:
    raise ValueError("No numeric field detected. Please set numeric_field_id to a numeric field's @id.")

if group_field_id is None:
    print("Warning: No obvious group field found. Grouping will be skipped.")

print(f"Using as numeric field: {numeric_field_id}")
if group_field_id is not None:
    print(f"Using as group field: {group_field_id}")

# EDA: Filter records for numeric_field > threshold
threshold = main_df[numeric_field_id].mean() if pd.notna(main_df[numeric_field_id].mean()) else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold]

print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f} (mean): {len(filtered_df)} records")
display(filtered_df.head())

# Normalize the field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"First 5 normalized '{numeric_field_id}' values:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional: Group and aggregate
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to the group field using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=20)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded a Croissant-structured biomedical record dataset via `mlcroissant` using only schema URLs and Croissant `@id` conventions.
- Explored the available record sets, fields, and demonstrated EDA referencing data elements strictly by `@id`.
- Visualized numerical variable distributions and provided a framework for further statistical and machine learning analyses on real-world clinical data.

**Next Steps:**
- Map the exact `@id` references for columns of specific analytical interest (see overview printout)
- Extend EDA for more features, associations, or statistical modelling
- Integrate Croissant datasets with ML workflows

_If you encounter empty DataFrames or no numeric columns, check the field `@id` assignments and inspect your record set and field summary in Section 2._